# 🌲 Minería de Datos — Random Forest con Comparación de Modelos


Este notebook estudia **Random Forest** y lo compara contra modelos supervisados trabajados previamente:

1. Regresión logística.
2. KNN.
3. Árbol de decisión.
4. Random Forest.

Usaremos el dataset **Titanic** para mantener continuidad.

## Objetivo general

Comprender cuándo Random Forest puede ser más conveniente que un árbol individual, KNN o regresión logística.

## Pregunta central

> ¿Qué modelo clasifica mejor la supervivencia en Titanic y cuál conviene usar según desempeño, interpretabilidad y robustez?

# 1. Comparación conceptual inicial

| Modelo | Enfoque | Ventaja principal | Limitación principal |
|---|---|---|---|
| Regresión logística | Probabilidad | Simple e interpretable | Puede quedarse corta con relaciones complejas |
| KNN | Distancia | Intuitivo | Requiere escalado |
| Árbol | Reglas | Muy interpretable | Puede sobreajustar |
| Random Forest | Muchos árboles | Robusto y potente | Menos interpretable |

La comparación será justa porque todos los modelos usarán el mismo dataset preparado y la misma división train/test.

# 2. Librerías

Importamos herramientas para:

- Manipular datos.
- Crear gráficos.
- Preparar datos.
- Entrenar modelos.
- Evaluar resultados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)

# 3. Carga del dataset Titanic

La variable objetivo es `Survived`:

- `0`: no sobrevivió.
- `1`: sobrevivió.

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dimensiones:", df.shape)
df.head()

# 4. Comprensión inicial

Revisamos la estructura del dataset y los valores faltantes antes de modelar.

In [ ]:
df.info()

In [ ]:
faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
}).sort_values("faltantes", ascending=False)

faltantes

# 5. Variable objetivo

Revisamos cuántos pasajeros sobrevivieron y cuántos no.

In [ ]:
conteo = df["Survived"].value_counts().sort_index()
porcentaje = df["Survived"].value_counts(normalize=True).sort_index() * 100

resumen_target = pd.DataFrame({
    "Survived": conteo.index,
    "descripcion": ["No sobrevivió", "Sobrevivió"],
    "conteo": conteo.values,
    "porcentaje": porcentaje.round(2).values
})

resumen_target

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(resumen_target["descripcion"], resumen_target["conteo"])
plt.title("Distribución de supervivencia")
plt.xlabel("Clase")
plt.ylabel("Número de pasajeros")
plt.show()

# 6. Exploración: sexo y clase

Estas variables suelen ser relevantes para explicar la supervivencia en Titanic.

In [ ]:
tabla_sex = pd.crosstab(df["Sex"], df["Survived"], normalize="index") * 100
tabla_sex.columns = ["No sobrevivió (%)", "Sobrevivió (%)"]
tabla_sex.round(2)

In [ ]:
tabla_sex.plot(kind="bar", figsize=(7,4))
plt.title("Supervivencia por sexo")
plt.xlabel("Sexo")
plt.ylabel("Porcentaje")
plt.xticks(rotation=0)
plt.show()

In [ ]:
tabla_clase = pd.crosstab(df["Pclass"], df["Survived"], normalize="index") * 100
tabla_clase.columns = ["No sobrevivió (%)", "Sobrevivió (%)"]
tabla_clase.round(2)

In [ ]:
tabla_clase.plot(kind="bar", figsize=(7,4))
plt.title("Supervivencia por clase")
plt.xlabel("Clase")
plt.ylabel("Porcentaje")
plt.xticks(rotation=0)
plt.show()

# 7. Preparación de datos

Variables usadas:

- Pclass
- Sex
- Age
- SibSp
- Parch
- Fare
- Embarked

Tratamiento:

- `Age`: imputación con mediana.
- `Embarked`: imputación con moda.
- `Sex`: codificación 0/1.
- `Embarked`: variables dummy.

No usamos `Cabin` por exceso de faltantes ni `Name`/`Ticket` por ser texto o códigos complejos.

In [ ]:
df_model = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

df_model["Age"] = df_model["Age"].fillna(df_model["Age"].median())
df_model["Embarked"] = df_model["Embarked"].fillna(df_model["Embarked"].mode()[0])
df_model["Sex"] = df_model["Sex"].map({"male": 0, "female": 1})

df_model = pd.get_dummies(df_model, columns=["Embarked"], drop_first=True)

print("Dimensiones:", df_model.shape)
df_model.head()

In [ ]:
df_model.isna().sum()

# 8. Separación X/y

- `X`: variables predictoras.
- `y`: variable objetivo.

In [ ]:
X = df_model.drop(columns=["Survived"])
y = df_model["Survived"]

print("X:", X.shape)
print("y:", y.shape)
print("Variables:", list(X.columns))

# 9. División train/test

Usamos la misma división para todos los modelos.

Esto permite una comparación justa.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

# 10. Función de evaluación

La función permite comparar todos los modelos con las mismas métricas:

- Accuracy.
- Precision.
- Recall.
- F1-score.

In [ ]:
def evaluar_clasificacion(nombre, y_real, y_pred):
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred),
        "recall": recall_score(y_real, y_pred),
        "f1": f1_score(y_real, y_pred)
    }

# 11. Modelo 1: Regresión logística

La regresión logística estima probabilidades para clasificación binaria.

Usamos escalado porque es un modelo lineal y puede beneficiarse de variables en escala comparable.

In [ ]:
modelo_logistica = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

modelo_logistica.fit(X_train, y_train)
pred_logistica = modelo_logistica.predict(X_test)

print(classification_report(y_test, pred_logistica, target_names=["No sobrevivió", "Sobrevivió"]))

# 12. Modelo 2: KNN

KNN clasifica usando vecinos cercanos.

Requiere escalado porque depende de distancias.

In [ ]:
modelo_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

modelo_knn.fit(X_train, y_train)
pred_knn = modelo_knn.predict(X_test)

print(classification_report(y_test, pred_knn, target_names=["No sobrevivió", "Sobrevivió"]))

# 13. Modelo 3: Árbol de decisión

El árbol genera reglas interpretables, pero puede sobreajustar.

In [ ]:
modelo_arbol = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

modelo_arbol.fit(X_train, y_train)
pred_arbol = modelo_arbol.predict(X_test)

print(classification_report(y_test, pred_arbol, target_names=["No sobrevivió", "Sobrevivió"]))

## Visualización del árbol

El árbol puede leerse como reglas IF-THEN.

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    modelo_arbol,
    feature_names=X.columns,
    class_names=["No sobrevivió", "Sobrevivió"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Árbol de decisión")
plt.show()

# 14. Modelo 4: Random Forest

Random Forest entrena muchos árboles y combina sus predicciones.

Esto reduce la dependencia de un único árbol y mejora la robustez.

Parámetros:

- `n_estimators`: número de árboles.
- `max_depth`: profundidad máxima.
- `random_state`: reproducibilidad.

In [ ]:
modelo_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

modelo_rf.fit(X_train, y_train)
pred_rf = modelo_rf.predict(X_test)

print(classification_report(y_test, pred_rf, target_names=["No sobrevivió", "Sobrevivió"]))

# 15. Comparación general de modelos

Comparamos los cuatro modelos con las mismas métricas.

In [ ]:
comparacion_modelos = pd.DataFrame([
    evaluar_clasificacion("Regresión logística", y_test, pred_logistica),
    evaluar_clasificacion("KNN k=5", y_test, pred_knn),
    evaluar_clasificacion("Árbol max_depth=3", y_test, pred_arbol),
    evaluar_clasificacion("Random Forest", y_test, pred_rf)
])

comparacion_modelos.sort_values("f1", ascending=False)

In [ ]:
metricas = ["accuracy", "precision", "recall", "f1"]

comparacion_modelos.set_index("modelo")[metricas].plot(kind="bar", figsize=(11,5))
plt.title("Comparación de modelos supervisados")
plt.ylabel("Valor")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.show()

# 16. Interpretación de comparación

Preguntas clave:

1. ¿Cuál modelo tiene mayor accuracy?
2. ¿Cuál tiene mayor recall?
3. ¿Cuál tiene mayor precision?
4. ¿Cuál tiene mejor F1?
5. ¿Cuál es más interpretable?
6. ¿Cuál depende más del preprocesamiento?
7. ¿Cuál recomendaría y por qué?

No siempre gana el modelo con mayor accuracy. La elección depende del objetivo.

# 17. Matrices de confusión comparadas

Las matrices de confusión muestran los aciertos y errores por clase.

In [ ]:
predicciones = {
    "Regresión logística": pred_logistica,
    "KNN": pred_knn,
    "Árbol": pred_arbol,
    "Random Forest": pred_rf
}

for nombre, pred in predicciones.items():
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["No sobrevivió", "Sobrevivió"])
    disp.plot()
    plt.title(f"Matriz de confusión - {nombre}")
    plt.show()

# 18. Importancia de variables en Random Forest

Random Forest permite calcular importancia de variables.

Importancia significa utilidad predictiva dentro del modelo, no causalidad.

In [ ]:
importancias = pd.DataFrame({
    "variable": X.columns,
    "importancia": modelo_rf.feature_importances_
}).sort_values("importancia", ascending=False)

importancias

In [ ]:
plt.figure(figsize=(9,5))
plt.barh(importancias["variable"], importancias["importancia"])
plt.gca().invert_yaxis()
plt.title("Importancia de variables - Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.show()

# 19. Tuning: número de árboles

Probamos distintos valores de `n_estimators`.

In [ ]:
valores_estimadores = [10, 50, 100, 200, 500]
resultados_estimadores = []

for n in valores_estimadores:
    modelo = RandomForestClassifier(n_estimators=n, random_state=42)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    resultados_estimadores.append({
        "n_estimators": n,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred)
    })

df_estimadores = pd.DataFrame(resultados_estimadores)
df_estimadores

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(df_estimadores["n_estimators"], df_estimadores["accuracy"], marker="o", label="Accuracy")
plt.plot(df_estimadores["n_estimators"], df_estimadores["f1"], marker="o", label="F1")
plt.title("Efecto de n_estimators")
plt.xlabel("Número de árboles")
plt.ylabel("Métrica")
plt.legend()
plt.grid(True)
plt.show()

# 20. Tuning: profundidad máxima

Probamos diferentes valores de `max_depth`.

In [ ]:
profundidades = [2, 3, 5, 8, 12, None]
resultados_depth = []

for depth in profundidades:
    modelo = RandomForestClassifier(
        n_estimators=200,
        max_depth=depth,
        random_state=42
    )
    modelo.fit(X_train, y_train)

    pred_train = modelo.predict(X_train)
    pred_test = modelo.predict(X_test)

    resultados_depth.append({
        "max_depth": str(depth),
        "accuracy_train": accuracy_score(y_train, pred_train),
        "accuracy_test": accuracy_score(y_test, pred_test),
        "f1_test": f1_score(y_test, pred_test)
    })

df_depth = pd.DataFrame(resultados_depth)
df_depth

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(df_depth["max_depth"], df_depth["accuracy_train"], marker="o", label="Train")
plt.plot(df_depth["max_depth"], df_depth["accuracy_test"], marker="o", label="Test")
plt.title("Efecto de max_depth en Random Forest")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 21. Random Forest regularizado

Entrenamos una versión más controlada:

- 200 árboles.
- profundidad máxima 5.
- mínimo 5 muestras por hoja.

In [ ]:
modelo_rf_reg = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

modelo_rf_reg.fit(X_train, y_train)
pred_rf_reg = modelo_rf_reg.predict(X_test)

print(classification_report(y_test, pred_rf_reg, target_names=["No sobrevivió", "Sobrevivió"]))

# 22. Comparación final incluyendo Random Forest regularizado

In [ ]:
comparacion_final = pd.DataFrame([
    evaluar_clasificacion("Regresión logística", y_test, pred_logistica),
    evaluar_clasificacion("KNN k=5", y_test, pred_knn),
    evaluar_clasificacion("Árbol max_depth=3", y_test, pred_arbol),
    evaluar_clasificacion("Random Forest base", y_test, pred_rf),
    evaluar_clasificacion("Random Forest regularizado", y_test, pred_rf_reg)
])

comparacion_final.sort_values("f1", ascending=False)

In [ ]:
comparacion_final.set_index("modelo")[metricas].plot(kind="bar", figsize=(12,5))
plt.title("Comparación final de modelos")
plt.ylabel("Valor")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.show()

# 23. Comparación conceptual final

| Modelo | Fortalezas | Debilidades |
|---|---|---|
| Regresión logística | Simple, probabilística, interpretable | Puede quedarse corta si la relación no es lineal |
| KNN | Intuitivo, basado en similitud | Sensible a escala y ruido |
| Árbol | Muy interpretable | Puede sobreajustar |
| Random Forest | Robusto, buen desempeño, importancia de variables | Menos interpretable, más costoso |

Una buena conclusión no se basa solo en una métrica. También considera interpretabilidad, estabilidad y contexto.

# 24. Taller final

## Parte A — Comparación conceptual

Responda:

1. ¿Qué diferencia hay entre regresión logística, KNN, árbol y Random Forest?
2. ¿Cuál modelo es más interpretable?
3. ¿Cuál modelo depende más del escalado?
4. ¿Cuál modelo usa votación de muchos modelos?

## Parte B — Comparación cuantitativa

Entrene y compare:

1. Regresión logística.
2. KNN.
3. Árbol de decisión.
4. Random Forest.

Use:

- accuracy,
- precision,
- recall,
- F1-score.

## Parte C — Matriz de confusión

Compare las matrices de confusión y responda:

1. ¿Cuál modelo comete menos falsos positivos?
2. ¿Cuál modelo comete menos falsos negativos?
3. ¿Cuál error considera más relevante en este contexto?

## Parte D — Random Forest

Analice:

1. Importancia de variables.
2. Número de árboles.
3. Profundidad máxima.
4. Posible overfitting.

## Parte E — Conclusión

Redacte mínimo 10 líneas indicando:

1. Qué modelo recomendaría.
2. Qué métrica usó para decidir.
3. Qué variables fueron más importantes.
4. Qué ventajas tiene Random Forest.
5. Qué limitaciones tiene.

# 25. Rúbrica sugerida

| Criterio | Puntaje |
|---|---:|
| Comparación conceptual de modelos | 0.8 |
| Preparación de datos | 0.6 |
| Entrenamiento de los modelos | 0.9 |
| Evaluación con métricas completas | 0.9 |
| Análisis de matrices de confusión | 0.6 |
| Importancia de variables e hiperparámetros | 0.8 |
| Conclusión técnica | 0.4 |
| **Total** | **5.0** |

## Cierre

Random Forest se entiende mejor cuando se compara contra otros modelos.

La pregunta final no es solamente:

> ¿Cuál modelo predice más?

Sino:

> ¿Cuál modelo conviene usar según desempeño, interpretación, costo y contexto?